Settings:
- `RUN_ID`: a run ID from `config/runs.py` (1-41; 26-41 are the ARR-October rebuttal runs), or a list such as `[7, 12]` for two runs on two GPUs.
- `SMOKE`: `True` runs a tiny subset for one epoch to check the pipeline; `False` runs the full experiment.

The dataset is found automatically under `/kaggle/input`, or in `data/` when run locally.

In [ ]:
RUN_ID = 7
SMOKE = False

In [ ]:
import glob
import os
import subprocess
import sys


def sh(*cmd: str, **kw: object) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)


if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = os.path.dirname(sorted(glob.glob("/kaggle/input/**/runner.py", recursive=True))[0])
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT = REPO_DIR

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Code {REPO_DIR} | output {OUT_ROOT}")

In [ ]:
from config.runs import describe
from runner import launch

print(describe(), "\n")
launch(RUN_ID, out_root=OUT_ROOT, smoke=SMOKE)

In [ ]:
import glob
import json

from IPython.display import Image, display

ids = [RUN_ID] if isinstance(RUN_ID, int) else RUN_ID
for f in sorted(glob.glob(f"{OUT_ROOT}/results/*_metrics.json")):
    with open(f) as fh:
        r = json.load(fh)
    if r["run_id"] not in ids or r["smoke"] != SMOKE:
        continue
    print(f"\n{r['tag']}  [{r['table']}] {r['row']}")
    print(
        f"  Macro-F1 {r['f1_macro'] * 100:.2f} | QWK {r['qwk']:.4f} | MAE {r['mae']:.4f} | "
        f"adjacent {r['adjacent_acc'] * 100:.2f}% | severe {r['severe_rate'] * 100:.2f}% | n={r['n_test']}"
    )
    print(
        "  Per-class F1: " + " / ".join(f"{c['name']} {c['f1'] * 100:.2f}" for c in r["per_class"])
    )
    print(f"  {r['train_seconds'] / 60:.1f} min on {r['environment']['gpu']}")
    png = f"{OUT_ROOT}/results/cm_{r['tag']}.png"
    if os.path.isfile(png):
        display(Image(png, width=480))

zips = sorted(glob.glob(f"{OUT_ROOT}/*codraft_runs_*.zip"), key=os.path.getmtime)
if zips:
    print(f"\nDownload this file: {zips[-1]} ({os.path.getsize(zips[-1]) / 2**20:.1f} MB)")
else:
    print("\nNo zip found; the launch cell may have failed before packaging.")